# ARC-AGI-3 Kaggle Submission Notebook — Offline Starter-Aligned Version

Notebook adaptado para Kaggle sin internet, usando `OperationMode.OFFLINE`, `environment_files` locales y un contrato de agente alineado al starter oficial (`GameAction`, `GameState`, `reasoning`, `RESET`).

Esta versión incorpora un agente V4 con control de estancamiento, reducción de loops, aprendizaje simple de acciones y heurística de cambio de estado.

## 1. Instalación de dependencias

In [ ]:
!pip install -q "arc-agi" "pillow<12"

## 2. Imports

In [ ]:
import random
from collections import defaultdict

import arc_agi
from arc_agi.base import OperationMode
from arcengine import GameAction, GameState


## 3. Configuración

In [ ]:
ENV_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
MAX_STEPS = 140
MAX_GAMES = None
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

## 4. Instanciación offline de Arcade

In [ ]:
arc = arc_agi.Arcade(
    operation_mode=OperationMode.OFFLINE,
    environments_dir=ENV_DIR
)

print("Arcade offline creada OK")
print("Entornos detectados:", len(arc.available_environments))

## 5. Detección automática de juegos

In [ ]:
detected_game_ids = []
seen_prefixes = set()

for env_info in arc.available_environments:
    full_id = env_info.game_id
    prefix = full_id.split("-")[0]
    if prefix not in seen_prefixes:
        detected_game_ids.append(prefix)
        seen_prefixes.add(prefix)

GAME_IDS = detected_game_ids[:MAX_GAMES] if MAX_GAMES is not None else detected_game_ids

print("Total detected game prefixes:", len(GAME_IDS))
print("Primeros 20:", GAME_IDS[:20])

## 6. Agente V4 alineado al starter oficial
Mantiene `GameAction`, `GameState`, `reasoning` y `RESET`, pero agrega control de estancamiento, aprendizaje simple de acciones y valoración de cambio de estado.

In [ ]:
class StarterAlignedAgentV4:
    MAX_ACTIONS = 140
    STAGNATION_LIMIT = 25

    def __init__(self, game_id):
        self.game_id = game_id
        self.coord_candidates = [
            (0, 0), (0, 15), (15, 0), (15, 15),
            (3, 3), (7, 7), (8, 4), (4, 8), (10, 10),
            (1, 8), (8, 1), (12, 6), (6, 12),
            (2, 2), (5, 10), (10, 5), (13, 13)
        ]
        self.reset()

    def reset(self):
        self.coord_index = 0
        self.action_counts = defaultdict(int)
        self.action_success = defaultdict(float)
        self.action_changefulness = defaultdict(float)
        self.state_counts = defaultdict(int)
        self.last_reward = 0.0
        self.steps_since_progress = 0
        self.last_state_key = None

    def state_key(self, latest_frame):
        if latest_frame is None:
            return "NONE"
        return repr(latest_frame)[:1200]

    def choose_coord(self):
        coord = self.coord_candidates[self.coord_index % len(self.coord_candidates)]
        self.coord_index += 1
        return coord

    def record_frame(self, latest_frame):
        key = self.state_key(latest_frame)
        self.state_counts[key] += 1
        return key

    def novelty_bonus(self, latest_frame):
        key = self.state_key(latest_frame)
        return 1.0 / (1.0 + self.state_counts[key])

    def is_done(self, latest_frame, action_count):
        if latest_frame is None:
            return False
        return any([
            latest_frame.state is GameState.WIN,
            action_count >= self.MAX_ACTIONS,
            self.steps_since_progress >= self.STAGNATION_LIMIT
        ])

    def choose_action(self, latest_frame):
        if latest_frame is not None and latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            action = GameAction.RESET
        else:
            available = [a for a in GameAction if a is not GameAction.RESET]
            novelty = self.novelty_bonus(latest_frame) if latest_frame is not None else 0.0

            def score(action):
                name = str(action)
                success_bonus = self.action_success[name]
                change_bonus = self.action_changefulness[name]
                usage_penalty = 0.22 * self.action_counts[name]
                return success_bonus + change_bonus + 0.10 * novelty - usage_penalty

            available.sort(key=score, reverse=True)
            pool = available[:max(1, len(available)//2)]
            action = random.choice(pool)

        if action.is_simple():
            action.reasoning = f"Picked {action.value} using reward + change heuristic"
        elif action.is_complex():
            x, y = self.choose_coord()
            action.set_data({"x": x, "y": y})
            action.reasoning = {
                "desired_action": action.value,
                "my_reason": "Systematic coordinate exploration guided by state change"
            }

        self.action_counts[str(action)] += 1
        return action

    def update(self, action, reward, previous_frame, latest_frame):
        action_name = str(action)

        prev_key = self.state_key(previous_frame)
        new_key = self.state_key(latest_frame)

        delta = reward - self.last_reward
        self.action_success[action_name] += delta

        state_changed = prev_key != new_key
        if state_changed:
            self.action_changefulness[action_name] += 0.5
        else:
            self.action_changefulness[action_name] -= 0.05

        if delta > 0 or state_changed:
            self.steps_since_progress = 0
        else:
            self.steps_since_progress += 1

        self.last_reward = reward
        self.record_frame(latest_frame)
        self.last_state_key = new_key


## 7. Utilidades para frames y step

In [ ]:
def extract_latest_frame(obs, info=None):
    if obs is None:
        return None

    if hasattr(obs, 'state'):
        return obs

    if isinstance(obs, dict):
        for key in ['latest_frame', 'frame', 'observation']:
            value = obs.get(key)
            if hasattr(value, 'state'):
                return value

    if isinstance(info, dict):
        for key in ['latest_frame', 'frame']:
            value = info.get(key)
            if hasattr(value, 'state'):
                return value

    return None


def parse_step_result(step_result):
    if not isinstance(step_result, tuple):
        return step_result, 0.0, False, False, {}

    if len(step_result) >= 5:
        return (
            step_result[0],
            step_result[1],
            bool(step_result[2]),
            bool(step_result[3]),
            step_result[4] if isinstance(step_result[4], dict) else {}
        )

    if len(step_result) == 4:
        obs, reward, done, info = step_result
        return obs, reward, bool(done), False, info if isinstance(info, dict) else {}

    if len(step_result) == 3:
        obs, reward, done = step_result
        return obs, reward, bool(done), False, {}

    if len(step_result) == 2:
        obs, reward = step_result
        return obs, reward, False, False, {}

    return step_result[0], 0.0, False, False, {}


## 8. Runner por juego (offline)

In [ ]:
def run_single_game_offline(game_id, max_steps=140):
    env = arc.make(game_id, render_mode=None)
    agent = StarterAlignedAgentV4(game_id=game_id)

    obs = None
    reward = 0.0
    terminated = False
    truncated = False
    info = {}
    latest_frame = None
    previous_frame = None
    trace = []

    for step in range(max_steps):
        latest_frame = extract_latest_frame(obs, info)

        if agent.is_done(latest_frame, step):
            break

        action = agent.choose_action(latest_frame)
        previous_frame = latest_frame

        step_result = env.step(action)
        obs, reward, terminated, truncated, info = parse_step_result(step_result)
        latest_frame = extract_latest_frame(obs, info)

        agent.update(action, reward, previous_frame, latest_frame)

        trace.append({
            "step": step,
            "action": str(action),
            "reward": reward,
            "terminated": terminated,
            "truncated": truncated,
        })

        if terminated or truncated:
            break

    return {
        "game_id": game_id,
        "steps": len(trace),
        "final_reward": reward,
        "scorecard": str(arc.get_scorecard())
    }


## 9. Runner batch

In [ ]:
def run_batch(game_ids, max_steps=140):
    results = []
    for game_id in game_ids:
        try:
            result = run_single_game_offline(game_id, max_steps=max_steps)
            results.append(result)
            print(f"OK  {game_id} | steps={result['steps']} | final_reward={result['final_reward']}")
        except Exception as e:
            results.append({
                "game_id": game_id,
                "error": str(e)
            })
            print(f"ERR {game_id} | {e}")
    return results


## 10. Ejecución

In [ ]:
batch_results = run_batch(GAME_IDS, max_steps=MAX_STEPS)
batch_results[:5]

## 11. Resumen agregado

In [ ]:
successful = [r for r in batch_results if 'error' not in r]
failed = [r for r in batch_results if 'error' in r]

summary = {
    "games_requested": len(GAME_IDS),
    "games_succeeded": len(successful),
    "games_failed": len(failed),
    "avg_steps": sum(r['steps'] for r in successful) / len(successful) if successful else None,
    "avg_final_reward": sum(r['final_reward'] for r in successful) / len(successful) if successful else None
}

summary

## 12. Nota final
Esta versión V4 mantiene el modo offline y el contrato del starter oficial, pero mejora el agente con control de estancamiento, reducción de loops, aprendizaje simple de acciones y heurística de cambio de estado.